# 03d — Shot-Type Classification: Ablation Study

Sequential ablations on ShuttleSet shot-type classification (17 classes).
Each group builds on the best config from the previous group.
Results saved to `results/ablations/{name}.json` — skip to §7 to view without retraining.

| Group | Ablation | Description |
|-------|----------|-------------|
| A | Skeleton representation | L2 vs L3 × dual-player vs single-player |
| B | Temporal window | Fixed T=32 vs variable (hit-frame boundaries) |
| C | Shuttle fusion | None vs graph node vs cross-attention |
| D | Classifier head | Linear vs 1-layer MLP vs 2-layer MLP |

## §1 — Setup & Colab Detection

In [ ]:
import os, sys, json, copy, zipfile, time
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'
    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.SS_SHUTTLES           = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_shuttles'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    _cfg.SS_CSV_ROOT   = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV  = _cfg.SS_CSV_ROOT / 'match.csv'
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'
    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Drive root: {DRIVE_ROOT}')
else:
    sys.path.insert(0, os.path.abspath('..'))
    DRIVE_ROOT = Path('..')
    print('Local run')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset as _Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, accuracy_score

from src.config import (
    get_config, FEATURE_DIMS, FEATURE_DIMS_WITH_HITTER,
    SS_SKELETONS_GDINO, SS_SHUTTLES, SS_SPLIT_JSON,
    SHOT_TYPES, NUM_SHOT_TYPES,
    NUM_NODES, NUM_JOINTS, MODELS_DIR, RESULTS_DIR,
)
from src.data.graph_builder import GraphBuilder
from src.data.dataset import ShuttleSetDataset
from src.models.stgcn_model import STGCN
from src.models.shuttle_cross_attn import ShuttleCrossAttention

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device: {device}')

ABLATION_DIR = RESULTS_DIR / 'ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
print(f'Ablation results: {ABLATION_DIR}')

## §2 — Shared Functions & Hyperparameters

All ablations use the same:
- Loss: cross-entropy (class-weighted)
- Optimizer: AdamW, lr=1e-3, cosine schedule
- Early stopping on **validation macro-F1** (patience=10) — 2 matches held out from training
- Final evaluation: macro-F1 + accuracy on the **test** set (2 held-out matches, never seen during training or early stopping)

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Shared hyperparameters ────────────────────────────────────────────────
SHOT_WINDOW   = 32
EPOCHS        = 80
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 10
N_CLASSES     = NUM_SHOT_TYPES  # 17

# ── Train / val / test split ─────────────────────────────────────────────
# Val = 2 matches from training set (early stopping); Test = held-out (final eval only)
with open(SS_SPLIT_JSON) as f:
    splits = json.load(f)

VAL_MATCH_NAMES = [
    'NG_Ka_Long_Angus_Jonatan_CHRISTIE_Malaysia_Masters_2020_QuarterFinals',
    'Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
]
TRAIN_MATCHES = set(splits['train']) - set(VAL_MATCH_NAMES)
VAL_MATCHES   = set(VAL_MATCH_NAMES)
TEST_MATCHES  = set(splits['held_out'])
print(f'Train: {len(TRAIN_MATCHES)} matches | Val: {len(VAL_MATCHES)} matches | Test: {len(TEST_MATCHES)} matches')

In [ ]:
def make_dataset(split_matches, feature_layer='L2', use_hitter=False,
                 use_shuttle=False, shuttle_fusion='graph',
                 variable_window=False):
    """Build a ShuttleSetDataset filtered to the given match set."""
    ds = ShuttleSetDataset(
        skeleton_dir=SS_SKELETONS_GDINO,
        shot_window=SHOT_WINDOW,
        feature_layer=feature_layer,
        load_shot_types=True,
        split=None,
        use_shuttle=use_shuttle,
        shuttle_dir=SS_SHUTTLES,
        use_hitter=use_hitter,
        variable_window=variable_window,
        shuttle_fusion=shuttle_fusion,
    )
    ds.samples = [s for s in ds.samples
                  if isinstance(s, dict) and Path(s.get('skel_dir', '')).name in split_matches]
    n_valid = sum(1 for s in ds.samples if s.get('shot_type_idx') is not None)
    print(f'  {len(ds)} samples ({n_valid} with labels)')
    return ds


class SinglePlayerWrapper(_Dataset):
    """Wraps a ShuttleSetDataset to return only the hitter's 17 joints."""
    def __init__(self, ds):
        self.ds = ds
        self.samples = ds.samples  # expose for class weight computation
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        sample = self.ds[idx]
        x, label = sample[0], sample[1]
        info = self.ds.samples[idx]
        hitter = info.get('hitter', 'top') if isinstance(info, dict) else 'top'
        if hitter == 'bottom':
            x = x[:, :, NUM_JOINTS:NUM_JOINTS*2]
        else:
            x = x[:, :, :NUM_JOINTS]
        return x, label


def collate_fn(batch):
    xs, labels = [], []
    for item in batch:
        xs.append(item[0])
        labels.append(item[1])
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long)


def collate_fn_shuttle(batch):
    xs, labels, shuttles = [], [], []
    for item in batch:
        xs.append(item[0])
        labels.append(item[1])
        if len(item) == 3:
            shuttles.append(item[2])
        else:
            shuttles.append(torch.zeros(2, SHOT_WINDOW))
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long), torch.stack(shuttles)


def compute_class_weights(dataset):
    from collections import Counter
    labels = [s.get('shot_type_idx') for s in dataset.samples
              if s.get('shot_type_idx') is not None]
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.ones(N_CLASSES, dtype=torch.float32)
    for cls_id, cnt in counts.items():
        weights[cls_id] = total / (len(counts) * cnt)
    return weights

In [ ]:
def build_encoder(in_channels, num_nodes, use_inter_player=True, single_player=False):
    """Build ST-GCN encoder + adjacency for the given config."""
    graph = GraphBuilder(
        use_inter_player=use_inter_player,
        single_player=single_player,
    )
    adj = graph.build_adjacency().to(device)
    n_nodes = NUM_JOINTS if single_player else NUM_NODES
    encoder = STGCN(
        in_channels=in_channels,
        num_nodes=n_nodes,
        adjacency=adj,
        num_layers=9,
        base_channels=64,
        embedding_dim=256,
        temporal_kernel=9,
        dropout=0.3,
    ).to(device)
    return encoder


def evaluate(encoder, head, ds, cross_attn_module=None):
    """Run batched evaluation, return (macro_f1, accuracy, y_true, y_pred)."""
    use_cross_attn = cross_attn_module is not None
    cfn = collate_fn_shuttle if use_cross_attn else collate_fn

    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=cfn,
    )

    encoder.eval(); head.eval()
    if use_cross_attn:
        cross_attn_module.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            if use_cross_attn:
                xb, yb, shuttle_b = batch
                shuttle_b = shuttle_b.to(device)
            else:
                xb, yb = batch

            valid = yb >= 0
            if not valid.any():
                continue

            xb_v = xb[valid].to(device)
            emb = encoder(xb_v)

            if use_cross_attn:
                emb = cross_attn_module(emb, shuttle_b[valid].to(device))

            preds = head(emb).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(yb[valid])

    if not all_preds:
        return 0.0, 0.0, np.array([]), np.array([])

    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    return macro_f1, accuracy, y_true, y_pred


def train_and_evaluate(name, train_ds, val_ds, test_ds, encoder, head,
                       cross_attn_module=None, epochs=EPOCHS, freeze_encoder=False):
    """
    Train encoder + head with CE loss. Early stop on val_ds macro-F1.
    Final evaluation on test_ds (never used for model selection).
    If freeze_encoder=True, only head parameters are trained.
    """
    use_cross_attn = cross_attn_module is not None
    cfn = collate_fn_shuttle if use_cross_attn else collate_fn

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True, collate_fn=cfn,
    )

    class_weights = compute_class_weights(train_ds).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    if freeze_encoder:
        for p in encoder.parameters():
            p.requires_grad_(False)
        encoder.eval()
        params = list(head.parameters())
        if use_cross_attn:
            params += list(cross_attn_module.parameters())
    else:
        params = list(encoder.parameters()) + list(head.parameters())
        if use_cross_attn:
            params += list(cross_attn_module.parameters())

    optimizer = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_f1 = -1.0
    no_improve = 0
    best_state = None
    history = {'train_loss': [], 'val_f1': [], 'val_acc': []}
    t0 = time.time()

    for epoch in range(epochs):
        # ── Train ─────────────────────────────────────────────────────
        if not freeze_encoder:
            encoder.train()
        head.train()
        if use_cross_attn:
            cross_attn_module.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in tqdm(train_loader, desc=f'[{name}] Epoch {epoch+1}/{epochs}', leave=False):
            if use_cross_attn:
                xb, yb, shuttle_b = batch
                shuttle_b = shuttle_b.to(device)
            else:
                xb, yb = batch

            valid = yb >= 0
            if not valid.any():
                continue

            xb = xb[valid].to(device)
            yb = yb[valid].to(device)

            emb = encoder(xb)

            if use_cross_attn:
                shuttle_b = shuttle_b[valid]
                emb = cross_attn_module(emb, shuttle_b)

            logits = head(emb)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            if use_cross_attn:
                torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)

        # ── Validate (for early stopping — NOT test set) ─────────────
        val_f1, val_acc, _, _ = evaluate(encoder, head, val_ds, cross_attn_module)

        history['train_loss'].append(avg_loss)
        history['val_f1'].append(val_f1)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | loss: {avg_loss:.4f} | val_f1: {val_f1:.4f}')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            no_improve = 0
            state = {
                'encoder': {k: v.cpu().clone() for k, v in encoder.state_dict().items()},
                'head': {k: v.cpu().clone() for k, v in head.state_dict().items()},
            }
            if use_cross_attn:
                state['cross_attn'] = {k: v.cpu().clone()
                                       for k, v in cross_attn_module.state_dict().items()}
            best_state = state
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1} (best val F1: {best_val_f1:.4f})')
                break

    train_time = time.time() - t0

    # ── Restore best checkpoint ───────────────────────────────────────
    if best_state:
        encoder.load_state_dict(best_state['encoder'])
        head.load_state_dict(best_state['head'])
        if use_cross_attn and 'cross_attn' in best_state:
            cross_attn_module.load_state_dict(best_state['cross_attn'])

    # ── Final evaluation on TEST set (unseen during training + early stopping)
    macro_f1, accuracy, y_true, y_pred = evaluate(encoder, head, test_ds, cross_attn_module)
    stopped_epoch = len(history['train_loss'])

    present_labels = sorted(set(y_true) | set(y_pred))
    label_names = [SHOT_TYPES[i] if i < len(SHOT_TYPES)
                   else f'type_{i}' for i in present_labels]

    print(f'\n  === {name} ====')
    print(f'  Test Accuracy:  {accuracy:.4f}')
    print(f'  Test Macro-F1:  {macro_f1:.4f}')
    print(f'  Best Val F1:    {best_val_f1:.4f}')
    print(f'  Epochs:         {stopped_epoch}')
    print(f'  Time:           {train_time:.0f}s')
    print(classification_report(
        y_true, y_pred, labels=present_labels,
        target_names=label_names, zero_division=0,
    ))

    report = classification_report(
        y_true, y_pred, labels=present_labels,
        target_names=label_names, zero_division=0, output_dict=True,
    )

    result = {
        'name': name,
        'accuracy': round(accuracy, 4),
        'macro_f1': round(macro_f1, 4),
        'best_val_f1': round(best_val_f1, 4),
        'stopped_epoch': stopped_epoch,
        'train_time_s': round(train_time, 1),
        'n_train': int(len(train_ds)),
        'n_val': int(len(val_ds)),
        'n_test': int(len(test_ds)),
        'n_test_labeled': int(len(y_true)),
        'history': history,
        'per_class': report,
    }
    save_path = ABLATION_DIR / f'{name}.json'
    with open(save_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved: {save_path}')

    ckpt_path = MODELS_DIR / f'ablation_{name}.pt'
    ckpt = {
        'encoder_state_dict': encoder.state_dict(),
        'head_state_dict': head.state_dict(),
        'name': name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
    }
    if use_cross_attn:
        ckpt['cross_attn_state_dict'] = cross_attn_module.state_dict()
    torch.save(ckpt, ckpt_path)
    print(f'  Checkpoint: {ckpt_path}')

    return result

## §3 — Group A: Skeleton Representation

Vary **feature layer** (L2 vs L3) and **player count** (dual 34-node vs single 17-node).
No shuttle, no hitter. Fixed T=32 window.

| # | Config | Features | Nodes | What it tests |
|---|--------|----------|-------|---------------|
| A1 | Dual L2 | 9-dim (xy + vel + court) | 34 | Baseline |
| A2 | Dual L3 | 12-dim (+ joint angles) | 34 | Do biomechanical features help? |
| A3 | Single L2 | 9-dim | 17 | Does opponent skeleton help? |
| A4 | Single L3 | 12-dim | 17 | Interaction: feature layer × player count |

In [ ]:
# ── A1: Dual-player, L2 (baseline) ────────────────────────────────────────
print('A1: Dual-player, L2 features (9-dim, 34 nodes)')
train_A1 = make_dataset(TRAIN_MATCHES, feature_layer='L2')
val_A1   = make_dataset(VAL_MATCHES, feature_layer='L2')
test_A1  = make_dataset(TEST_MATCHES, feature_layer='L2')

encoder_A1 = build_encoder(in_channels=FEATURE_DIMS['L2'], num_nodes=NUM_NODES)
head_A1 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A1.parameters()) + sum(p.numel() for p in head_A1.parameters()):,}')

result_A1 = train_and_evaluate('A1_dual_L2', train_A1, val_A1, test_A1, encoder_A1, head_A1)

In [ ]:
# ── A2: Dual-player, L3 ───────────────────────────────────────────────────
print('A2: Dual-player, L3 features (12-dim, 34 nodes)')
train_A2 = make_dataset(TRAIN_MATCHES, feature_layer='L3')
val_A2   = make_dataset(VAL_MATCHES, feature_layer='L3')
test_A2  = make_dataset(TEST_MATCHES, feature_layer='L3')

encoder_A2 = build_encoder(in_channels=FEATURE_DIMS['L3'], num_nodes=NUM_NODES)
head_A2 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A2.parameters()) + sum(p.numel() for p in head_A2.parameters()):,}')

result_A2 = train_and_evaluate('A2_dual_L3', train_A2, val_A2, test_A2, encoder_A2, head_A2)

In [ ]:
# ── A3: Single-player, L2 ─────────────────────────────────────────────────
print('A3: Single-player, L2 features (9-dim, 17 nodes)')
train_A3_raw = make_dataset(TRAIN_MATCHES, feature_layer='L2')
val_A3_raw   = make_dataset(VAL_MATCHES, feature_layer='L2')
test_A3_raw  = make_dataset(TEST_MATCHES, feature_layer='L2')
train_A3 = SinglePlayerWrapper(train_A3_raw)
val_A3   = SinglePlayerWrapper(val_A3_raw)
test_A3  = SinglePlayerWrapper(test_A3_raw)

encoder_A3 = build_encoder(in_channels=FEATURE_DIMS['L2'], num_nodes=NUM_JOINTS, single_player=True)
head_A3 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A3.parameters()) + sum(p.numel() for p in head_A3.parameters()):,}')

result_A3 = train_and_evaluate('A3_single_L2', train_A3, val_A3, test_A3, encoder_A3, head_A3)

In [ ]:
# ── A4: Single-player, L3 ─────────────────────────────────────────────────
print('A4: Single-player, L3 features (12-dim, 17 nodes)')
train_A4_raw = make_dataset(TRAIN_MATCHES, feature_layer='L3')
val_A4_raw   = make_dataset(VAL_MATCHES, feature_layer='L3')
test_A4_raw  = make_dataset(TEST_MATCHES, feature_layer='L3')
train_A4 = SinglePlayerWrapper(train_A4_raw)
val_A4   = SinglePlayerWrapper(val_A4_raw)
test_A4  = SinglePlayerWrapper(test_A4_raw)

encoder_A4 = build_encoder(in_channels=FEATURE_DIMS['L3'], num_nodes=NUM_JOINTS, single_player=True)
head_A4 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A4.parameters()) + sum(p.numel() for p in head_A4.parameters()):,}')

result_A4 = train_and_evaluate('A4_single_L3', train_A4, val_A4, test_A4, encoder_A4, head_A4)

In [ ]:
# ── Pick Group A winner ───────────────────────────────────────────────────
grpA = {
    'A1_dual_L2': result_A1, 'A2_dual_L3': result_A2,
    'A3_single_L2': result_A3, 'A4_single_L3': result_A4,
}
best_A_name = max(grpA, key=lambda k: grpA[k]['macro_f1'])
best_A = grpA[best_A_name]

print('Group A results:')
for name, r in grpA.items():
    marker = ' <-- best' if name == best_A_name else ''
    print(f'  {name:<20} Acc={r["accuracy"]:.4f}  F1={r["macro_f1"]:.4f}{marker}')

# Determine config for Group B
BEST_FEATURE = 'L3' if 'L3' in best_A_name else 'L2'
BEST_SINGLE  = 'single' in best_A_name
print(f'\nGroup B will use: feature_layer={BEST_FEATURE}, single_player={BEST_SINGLE}')

# Save winners for §7 standalone display
_winners = {'A': best_A_name}
with open(ABLATION_DIR / '_winners.json', 'w') as f:
    json.dump(_winners, f)

## §4 — Group B: Temporal Window

Using the best skeleton config from Group A, compare:
- **Fixed T=32**: symmetric 16-frame window around hit frame
- **Variable**: prev shot hit → next shot hit, resampled to T=32

The Group A winner result is reused as B1 (no retraining needed).

In [ ]:
# B1 = Group A winner (fixed window) — already trained
result_B1 = best_A
print(f'B1 (fixed window): reusing {best_A_name} — F1={result_B1["macro_f1"]:.4f}')

# ── B2: Variable window ───────────────────────────────────────────────────
print(f'\nB2: Variable window ({BEST_FEATURE}, single={BEST_SINGLE})')
train_B2_raw = make_dataset(TRAIN_MATCHES, feature_layer=BEST_FEATURE, variable_window=True)
val_B2_raw   = make_dataset(VAL_MATCHES, feature_layer=BEST_FEATURE, variable_window=True)
test_B2_raw  = make_dataset(TEST_MATCHES, feature_layer=BEST_FEATURE, variable_window=True)

if BEST_SINGLE:
    train_B2 = SinglePlayerWrapper(train_B2_raw)
    val_B2   = SinglePlayerWrapper(val_B2_raw)
    test_B2  = SinglePlayerWrapper(test_B2_raw)
    encoder_B2 = build_encoder(in_channels=FEATURE_DIMS[BEST_FEATURE], num_nodes=NUM_JOINTS, single_player=True)
else:
    train_B2 = train_B2_raw
    val_B2   = val_B2_raw
    test_B2  = test_B2_raw
    encoder_B2 = build_encoder(in_channels=FEATURE_DIMS[BEST_FEATURE], num_nodes=NUM_NODES)

head_B2 = nn.Linear(256, N_CLASSES).to(device)
result_B2 = train_and_evaluate('B2_variable_window', train_B2, val_B2, test_B2, encoder_B2, head_B2)

# ── Pick Group B winner ───────────────────────────────────────────────────
if result_B2['macro_f1'] > result_B1['macro_f1']:
    best_B = result_B2
    best_B_name = 'B2_variable_window'
    BEST_WINDOW = True
    print(f'\nWinner: Variable window (F1={result_B2["macro_f1"]:.4f} > {result_B1["macro_f1"]:.4f})')
else:
    best_B = result_B1
    best_B_name = best_A_name
    BEST_WINDOW = False
    print(f'\nWinner: Fixed window (F1={result_B1["macro_f1"]:.4f} >= {result_B2["macro_f1"]:.4f})')

print(f'Group C will use: {BEST_FEATURE}, single={BEST_SINGLE}, variable_window={BEST_WINDOW}')

# Update winners
_winners['B'] = best_B_name
with open(ABLATION_DIR / '_winners.json', 'w') as f:
    json.dump(_winners, f)

## §5 — Group C: Shuttle Fusion

Using the best config from Groups A+B, compare shuttle integration methods:
- **C1 (no shuttle)**: Group B winner — already trained
- **C2 (graph node)**: shuttle as virtual node 35 in skeleton graph
- **C3 (cross-attention)**: separate shuttle TCN + cross-attention fusion

In [ ]:
# C1 = Group B winner (no shuttle) — already trained
result_C1 = best_B
print(f'C1 (no shuttle): reusing B winner — F1={result_C1["macro_f1"]:.4f}')

# Common dataset kwargs for Group C
_c_kwargs = dict(feature_layer=BEST_FEATURE, variable_window=BEST_WINDOW)

# ── C2: Shuttle as graph node ─────────────────────────────────────────────
print(f'\nC2: + Shuttle (graph node)')
train_C2 = make_dataset(TRAIN_MATCHES, use_shuttle=True, shuttle_fusion='graph', **_c_kwargs)
val_C2   = make_dataset(VAL_MATCHES, use_shuttle=True, shuttle_fusion='graph', **_c_kwargs)
test_C2  = make_dataset(TEST_MATCHES, use_shuttle=True, shuttle_fusion='graph', **_c_kwargs)

in_ch_C2 = FEATURE_DIMS[BEST_FEATURE]
graph_C2 = GraphBuilder(use_inter_player=True, single_player=False, use_shuttle=True)
adj_C2 = graph_C2.build_adjacency().to(device)
encoder_C2 = STGCN(
    in_channels=in_ch_C2, num_nodes=35, adjacency=adj_C2,
    num_layers=9, base_channels=64, embedding_dim=256,
    temporal_kernel=9, dropout=0.3,
).to(device)
head_C2 = nn.Linear(256, N_CLASSES).to(device)
result_C2 = train_and_evaluate('C2_shuttle_graph', train_C2, val_C2, test_C2, encoder_C2, head_C2)

# ── C3: Shuttle via cross-attention ───────────────────────────────────────
print(f'\nC3: + Shuttle (cross-attention)')
train_C3 = make_dataset(TRAIN_MATCHES, use_shuttle=True, shuttle_fusion='cross_attn', **_c_kwargs)
val_C3   = make_dataset(VAL_MATCHES, use_shuttle=True, shuttle_fusion='cross_attn', **_c_kwargs)
test_C3  = make_dataset(TEST_MATCHES, use_shuttle=True, shuttle_fusion='cross_attn', **_c_kwargs)

in_ch_C3 = FEATURE_DIMS[BEST_FEATURE]
encoder_C3 = build_encoder(in_channels=in_ch_C3, num_nodes=NUM_NODES)
cross_attn_C3 = ShuttleCrossAttention(d_skel=256, d_shuttle=128, nhead=4).to(device)
head_C3 = nn.Linear(256, N_CLASSES).to(device)
result_C3 = train_and_evaluate('C3_shuttle_crossattn', train_C3, val_C3, test_C3,
                                encoder_C3, head_C3, cross_attn_module=cross_attn_C3)

# ── Pick Group C winner ───────────────────────────────────────────────────
grpC = {'C1_no_shuttle': result_C1, 'C2_shuttle_graph': result_C2, 'C3_shuttle_crossattn': result_C3}
best_C_name = max(grpC, key=lambda k: grpC[k]['macro_f1'])
best_C = grpC[best_C_name]
# Map C1 back to the actual B winner's result name
if best_C_name == 'C1_no_shuttle':
    best_C_name = best_B_name
print(f'\nGroup C results:')
for name, r in grpC.items():
    marker = ' <-- best' if r is best_C else ''
    print(f'  {name:<25} Acc={r["accuracy"]:.4f}  F1={r["macro_f1"]:.4f}{marker}')

BEST_C_ENCODER_NAME = best_C_name

# Update winners
_winners['C'] = best_C_name
with open(ABLATION_DIR / '_winners.json', 'w') as f:
    json.dump(_winners, f)

## §6 — Group D: Classifier Head

Using the best encoder from Groups A+B+C (**frozen**), compare classifier heads:
- **D1 (linear)**: `Linear(256, 17)` — Group C winner, already trained
- **D2 (MLP-1)**: `Linear(256, 128) → ReLU → Dropout(0.3) → Linear(128, 17)`
- **D3 (MLP-2)**: `Linear(256, 128) → ReLU → Dropout(0.3) → Linear(128, 64) → ReLU → Dropout(0.3) → Linear(64, 17)`

The encoder is frozen to isolate the head's contribution. Only head parameters are optimised.
Note: head dropout (0.3) stacks with ST-GCN dropout (0.3) — if MLPs underperform,
combined regularisation may be too aggressive.

In [ ]:
# D1 = Group C winner (linear head) — already trained
result_D1 = best_C
print(f'D1 (linear): reusing C winner — F1={result_D1["macro_f1"]:.4f}')

# Load C winner's encoder checkpoint for freezing
_c_ckpt_path = MODELS_DIR / f'ablation_{best_C_name}.pt'
_c_ckpt = torch.load(_c_ckpt_path, map_location=device, weights_only=True)

# Determine full config from previous winners
BEST_SHUTTLE = 'graph' if 'graph' in best_C_name else ('cross_attn' if 'cross' in best_C_name else None)
_d_kwargs = dict(feature_layer=BEST_FEATURE, variable_window=BEST_WINDOW)
_use_shuttle = BEST_SHUTTLE is not None

def _make_d_datasets():
    """Build train/val/test datasets with the full A+B+C winner config."""
    if _use_shuttle:
        kw = dict(use_shuttle=True, shuttle_fusion=BEST_SHUTTLE, **_d_kwargs)
    else:
        kw = dict(**_d_kwargs)
    tr = make_dataset(TRAIN_MATCHES, **kw)
    va = make_dataset(VAL_MATCHES, **kw)
    te = make_dataset(TEST_MATCHES, **kw)
    if BEST_SINGLE:
        tr = SinglePlayerWrapper(tr)
        va = SinglePlayerWrapper(va)
        te = SinglePlayerWrapper(te)
    return tr, va, te

def _make_d_encoder_frozen():
    """Build encoder matching C winner, load weights, freeze."""
    if _use_shuttle and BEST_SHUTTLE == 'graph':
        graph = GraphBuilder(use_inter_player=True, single_player=False, use_shuttle=True)
        adj = graph.build_adjacency().to(device)
        enc = STGCN(in_channels=FEATURE_DIMS[BEST_FEATURE], num_nodes=35,
                     adjacency=adj, num_layers=9, base_channels=64,
                     embedding_dim=256, temporal_kernel=9, dropout=0.3).to(device)
        ca = None
    else:
        n_nodes = NUM_JOINTS if BEST_SINGLE else NUM_NODES
        enc = build_encoder(in_channels=FEATURE_DIMS[BEST_FEATURE],
                            num_nodes=n_nodes, single_player=BEST_SINGLE)
        ca = None
        if _use_shuttle and BEST_SHUTTLE == 'cross_attn':
            ca = ShuttleCrossAttention(d_skel=256, d_shuttle=128, nhead=4).to(device)
            ca.load_state_dict(_c_ckpt['cross_attn_state_dict'])

    enc.load_state_dict(_c_ckpt['encoder_state_dict'])
    return enc, ca

# ── D2: 1-layer MLP (frozen encoder) ─────────────────────────────────────
print(f'\nD2: MLP-1 head (256 → 128 → 17), frozen encoder from {best_C_name}')
train_D2, val_D2, test_D2 = _make_d_datasets()
encoder_D2, cross_attn_D2 = _make_d_encoder_frozen()
head_D2 = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, N_CLASSES),
).to(device)
result_D2 = train_and_evaluate('D2_mlp1', train_D2, val_D2, test_D2,
                                encoder_D2, head_D2,
                                cross_attn_module=cross_attn_D2,
                                freeze_encoder=True)

# ── D3: 2-layer MLP (frozen encoder) ─────────────────────────────────────
print(f'\nD3: MLP-2 head (256 → 128 → 64 → 17), frozen encoder from {best_C_name}')
train_D3, val_D3, test_D3 = _make_d_datasets()
encoder_D3, cross_attn_D3 = _make_d_encoder_frozen()
head_D3 = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, N_CLASSES),
).to(device)
result_D3 = train_and_evaluate('D3_mlp2', train_D3, val_D3, test_D3,
                                encoder_D3, head_D3,
                                cross_attn_module=cross_attn_D3,
                                freeze_encoder=True)

# ── Pick Group D winner ───────────────────────────────────────────────────
grpD = {'D1_linear': result_D1, 'D2_mlp1': result_D2, 'D3_mlp2': result_D3}
best_D_name = max(grpD, key=lambda k: grpD[k]['macro_f1'])
print(f'\nGroup D results:')
for name, r in grpD.items():
    marker = ' <-- best' if name == best_D_name else ''
    print(f'  {name:<20} Acc={r["accuracy"]:.4f}  F1={r["macro_f1"]:.4f}{marker}')

# Save winner
_winners['D'] = best_D_name
with open(ABLATION_DIR / '_winners.json', 'w') as f:
    json.dump(_winners, f)
print(f'Saved _winners["D"] = {best_D_name}')

## §7 — Results Comparison

Load all saved ablation JSONs and render grouped comparison tables + chart.
**This section can be run standalone** (restart kernel → run §1–§2 → run §7).

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

try:
    _abl_dir = ABLATION_DIR
except NameError:
    from src.config import RESULTS_DIR
    _abl_dir = RESULTS_DIR / 'ablations'

# ── Load winners from previous groups ──────────────────────────────────────
_winners_path = _abl_dir / '_winners.json'
if _winners_path.exists():
    with open(_winners_path) as f:
        _w = json.load(f)
else:
    _w = {}

win_A = _w.get('A', 'A1_dual_L2')
win_B = _w.get('B', win_A)
win_C = _w.get('C', win_B)
win_D = _w.get('D', win_C)

# ── Define groups (baselines resolved dynamically) ─────────────────────────
groups = [
    ('Group A: Skeleton Representation', [
        ('A1_dual_L2',   'Dual-player, L2 (9-dim)'),
        ('A2_dual_L3',   'Dual-player, L3 (12-dim)'),
        ('A3_single_L2', 'Single-player, L2 (9-dim)'),
        ('A4_single_L3', 'Single-player, L3 (12-dim)'),
    ]),
    ('Group B: Temporal Window', [
        (win_A,                'Fixed T=32 (= A winner)'),
        ('B2_variable_window', 'Variable (hit-frame boundaries)'),
    ]),
    ('Group C: Shuttle Fusion', [
        (win_B,                 'No shuttle (= B winner)'),
        ('C2_shuttle_graph',    '+ Shuttle (graph node)'),
        ('C3_shuttle_crossattn', '+ Shuttle (cross-attention)'),
    ]),
    ('Group D: Classifier Head', [
        (win_C,     'Linear (= C winner)'),
        ('D2_mlp1', 'MLP-1 (256→128→17)'),
        ('D3_mlp2', 'MLP-2 (256→128→64→17)'),
    ]),
]

# Load all results
all_names = set()
for _, items in groups:
    for name, _ in items:
        all_names.add(name)

loaded = {}
for name in all_names:
    p = _abl_dir / f'{name}.json'
    if p.exists():
        with open(p) as f:
            loaded[name] = json.load(f)

# ── Grouped tables ─────────────────────────────────────────────────────────
for group_title, items in groups:
    print(f'\n{"=" * 80}')
    print(f'{group_title}')
    print(f'{"=" * 80}')
    print(f'{"":<4} {"Ablation":<38} {"Accuracy":>10} {"Macro-F1":>10} {"Epochs":>8}')
    print(f'{"-" * 80}')
    best_f1 = max((loaded[n]['macro_f1'] for n, _ in items if n in loaded), default=0)
    for i, (name, display) in enumerate(items, 1):
        if name not in loaded:
            print(f'{i:<4} {display:<38} {"(not run)":>10}')
        else:
            r = loaded[name]
            marker = ' *' if r['macro_f1'] == best_f1 else ''
            print(f'{i:<4} {display:<38} {r["accuracy"]:>10.4f} {r["macro_f1"]:>10.4f} {r["stopped_epoch"]:>8}{marker}')

# ── Combined bar chart ──────────────────────────────────────────────────────
n_groups = len(groups)
fig, axes = plt.subplots(1, n_groups, figsize=(5 * n_groups + 1, 5), sharey=True)
group_colors = [
    ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'],
    ['#3498db', '#e67e22'],
    ['#3498db', '#e67e22', '#9b59b6'],
    ['#3498db', '#e67e22', '#9b59b6'],
]

for ax, (group_title, items), colors in zip(axes, groups, group_colors):
    names_plot, f1_plot, acc_plot = [], [], []
    for name, display in items:
        if name in loaded:
            clean = display.replace(' (= A winner)', '').replace(' (= B winner)', '').replace(' (= C winner)', '')
            names_plot.append(clean)
            f1_plot.append(loaded[name]['macro_f1'])
            acc_plot.append(loaded[name]['accuracy'])

    if not names_plot:
        ax.set_title(group_title.split(':')[1].strip())
        ax.text(0.5, 0.5, 'No results', ha='center', va='center', transform=ax.transAxes)
        continue

    x = np.arange(len(names_plot))
    w = 0.35
    ax.bar(x - w/2, acc_plot, w, label='Accuracy', color=colors[:len(x)], alpha=0.7)
    bars = ax.bar(x + w/2, f1_plot, w, label='Macro-F1', color=colors[:len(x)], edgecolor='black', lw=0.8)
    for bar, val in zip(bars, f1_plot):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names_plot, rotation=25, ha='right', fontsize=8)
    ax.set_title(group_title.split(':')[1].strip(), fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

axes[0].set_ylabel('Score')
axes[0].legend(fontsize=8)
fig.suptitle('Shot-Type Classification — Sequential Ablations', fontsize=12, fontweight='bold')
plt.tight_layout()

fig_path = _abl_dir / 'ablation_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# ── Training curves overlay ───────────────────────────────────────────────
all_ablation_names = []
for _, items in groups:
    for name, _ in items:
        if name not in [n for n, _ in all_ablation_names]:
            display = [d for n, d in items if n == name][0]
            all_ablation_names.append((name, display))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_all = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#e67e22', '#1abc9c', '#f39c12']

ci = 0
for name, display in all_ablation_names:
    if name in loaded and 'history' in loaded[name]:
        h = loaded[name]['history']
        c = colors_all[ci % len(colors_all)]
        losses = h['train_loss']
        ax1.plot(range(1, len(losses)+1), losses, label=display, color=c, linewidth=1.5)
        if 'val_f1' in h:
            val_f1 = h['val_f1']
            ax2.plot(range(1, len(val_f1)+1), val_f1, label=display, color=c, linewidth=1.5)
        ci += 1

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss (CE, class-weighted)')
ax1.set_title('Training Loss')
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Macro-F1')
ax2.set_title('Validation Macro-F1')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

fig.suptitle('Training Curves — All Ablations', fontsize=12, fontweight='bold')
plt.tight_layout()

fig_path2 = _abl_dir / 'ablation_training_curves.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path2}')